# How the language-model data was made

This notebook produced every file in this folder. You do not need to run it: the course
notebooks 16-19 read the cached files and run offline. It is here so that every number in
Part III can be traced back to a request, a prompt and a model.

What it does, in order:

1. Downloads three public datasets: a non-medical slice of MMLU (multiple-choice exam
   questions, MIT licence), the HotpotQA distractor validation set (questions with 2 supporting
   and 8 distractor Wikipedia passages, CC BY-SA 4.0), and PubChem's periodic table (public
   domain, U.S. National Library of Medicine).
2. Asks `gpt-4.1-mini` on Azure OpenAI for: answer-letter log-probabilities, a stated
   confidence, ten sampled answers, and three re-runs that change the procedure (same prompt
   again, shuffled options, a different prompt) for each multiple-choice question; a profile
   of each chemical element as tagged claims, plus five resampled profiles; and a short answer
   to each HotpotQA question from its top 1, 2, 3, 5 and 10 retrieved passages.
3. Embeds HotpotQA questions and passages with `text-embedding-3-large` to score retrieval.
4. Grades everything automatically against the datasets' own answers and writes tidy CSV files.

To re-run it you need an Azure OpenAI resource. Set `AZURE_OPENAI_ENDPOINT` (and optionally
`AZURE_OPENAI_CHAT_DEPLOYMENT`, `AZURE_OPENAI_EMBED_DEPLOYMENT`); authentication uses
`AZURE_OPENAI_API_KEY` if set, otherwise a token from the Azure CLI (`az login`). Every request
is cached on disk under a hash of its exact body (`LLM_CACHE_DIR`, default
`~/.cache/conformal_course_llm`), so a re-run reuses earlier answers and costs nothing. A
hosted model is not bit-for-bit deterministic, so a run without the cache will produce
slightly different numbers; the committed files are the reference.

In [1]:
import gzip
import hashlib
import json
import math
import os
import re
import string
import subprocess
import threading
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import requests

RANDOM_SEED = 20260707
LIMIT = int(os.environ.get("LLM_GEN_LIMIT", "0"))      # >0: tiny test run
OUT = Path(os.environ.get("LLM_OUT_DIR", "."))         # where the tidy files go
CACHE = Path(os.environ.get("LLM_CACHE_DIR", Path.home() / ".cache" / "conformal_course_llm"))
SRC = CACHE / "sources"
for d in (OUT, CACHE / "requests", SRC):
    d.mkdir(parents=True, exist_ok=True)

CHAT = os.environ.get("AZURE_OPENAI_CHAT_DEPLOYMENT", "gpt-4.1-mini")
EMBED = os.environ.get("AZURE_OPENAI_EMBED_DEPLOYMENT", "text-embedding-3-large")
API_VERSION = "2025-04-01-preview"
ENDPOINT = os.environ.get("AZURE_OPENAI_ENDPOINT", "").rstrip("/")   # never printed
CACHE_ONLY = os.environ.get("LLM_CACHE_ONLY") == "1"   # rebuild from the cache, never send a request

N_MMLU_PER_SUBJECT = 90
N_HOTPOT = 800
K_GRID = (1, 2, 3, 5, 10)
N_ELEMENT_RESAMPLES = 5
N_MCQ_SAMPLES = 10
if LIMIT:
    N_MMLU_PER_SUBJECT, N_HOTPOT = 2, LIMIT
print(f"chat model: {CHAT} | embedding model: {EMBED} | test limit: {LIMIT or 'off'} | "
      f"cache only: {CACHE_ONLY}")

chat model: gpt-4.1-mini | embedding model: text-embedding-3-large | test limit: off | cache only: True


## The request layer

In [2]:
_token = {"value": None, "time": 0.0}
_lock = threading.Lock()
USAGE = Counter()
RESPONSE_MODELS = Counter()   # model version the API reported, per (kind, model)


def auth_headers(force_refresh=False):
    """API key if provided, otherwise an Azure CLI token, refreshed every 10 minutes and on 401.

    `az account get-access-token` can return a cached token with only minutes left, so the
    refresh interval is short and a 401 forces a new token.
    """
    if os.environ.get("AZURE_OPENAI_API_KEY"):
        return {"api-key": os.environ["AZURE_OPENAI_API_KEY"]}
    with _lock:
        if force_refresh or time.time() - _token["time"] > 600:
            _token["value"] = subprocess.run(
                ["az", "account", "get-access-token", "--resource",
                 "https://cognitiveservices.azure.com", "--query", "accessToken", "-o", "tsv"],
                capture_output=True, text=True, check=True).stdout.strip()
            _token["time"] = time.time()
        return {"Authorization": f"Bearer {_token['value']}"}


def call(kind, body):
    """POST one request (kind = 'chat' or 'embed'), cached on disk by a hash of its exact body."""
    deployment = CHAT if kind == "chat" else EMBED
    key = hashlib.sha256(json.dumps({"kind": kind, "deployment": deployment, "body": body},
                                    sort_keys=True).encode()).hexdigest()
    path = CACHE / "requests" / f"{key}.json"
    if path.exists():
        data = json.loads(path.read_text())
        with _lock:
            USAGE["cached"] += 1
            if not data.get("filtered"):
                RESPONSE_MODELS[f"{kind}: {data.get('model')}"] += 1
        return data
    if CACHE_ONLY:
        raise RuntimeError("cache miss with LLM_CACHE_ONLY=1: this run would send a new request")
    if not ENDPOINT:
        raise RuntimeError("set AZURE_OPENAI_ENDPOINT to send new requests")
    route = "chat/completions" if kind == "chat" else "embeddings"
    url = f"{ENDPOINT}/openai/deployments/{deployment}/{route}?api-version={API_VERSION}"
    for attempt in range(10):
        try:
            r = requests.post(url, headers={**auth_headers(), "Content-Type": "application/json"},
                              json=body, timeout=120)
        except (requests.ConnectionError, requests.Timeout):
            # Network blips (DNS, resets) are retried like rate limits; the error text is not
            # printed because it would contain the endpoint.
            time.sleep(min(60, 2 ** attempt + np.random.rand()))
            continue
        if r.status_code == 200:
            data = r.json()
            path.write_text(json.dumps(data))
            u = data.get("usage", {})
            with _lock:
                USAGE["requests"] += 1
                RESPONSE_MODELS[f"{kind}: {data.get('model')}"] += 1
                USAGE[f"{kind}_prompt_tokens"] += u.get("prompt_tokens", 0)
                USAGE[f"{kind}_completion_tokens"] += u.get("completion_tokens", 0)
            return data
        if r.status_code == 401:
            auth_headers(force_refresh=True)
            continue
        if r.status_code in (408, 429, 500, 502, 503, 504):
            time.sleep(min(60, 2 ** attempt + np.random.rand()))
            continue
        error = r.json().get("error", {}) if "json" in r.headers.get("Content-Type", "") else {}
        if r.status_code == 400 and (error.get("code") == "content_filter"
                                     or "content management policy" in str(error.get("message", ""))):
            # Azure's content filter refused the prompt. Cache the refusal so a re-run does not
            # ask again; the question is dropped downstream and counted.
            data = {"filtered": True}
            path.write_text(json.dumps(data))
            with _lock:
                USAGE["filtered"] += 1
            return data
        raise RuntimeError(f"HTTP {r.status_code} from the {kind} deployment: "
                           f"{error.get('code')} {str(error.get('message', ''))[:200]}")
    raise RuntimeError(f"too many retries on the {kind} deployment")


def filtered(resp):
    return bool(resp.get("filtered"))


def run_all(jobs, fn, workers=16):
    """Apply fn to every job with a thread pool, keeping the input order."""
    with ThreadPoolExecutor(workers) as pool:
        return list(pool.map(fn, jobs))


def download(url, name):
    path = SRC / name
    if not path.exists():
        r = requests.get(url, timeout=300)
        r.raise_for_status()
        path.write_bytes(r.content)
    return path

## 1. Multiple-choice questions (MMLU, non-medical subjects)

In [3]:
SUBJECTS = [
    "abstract_algebra", "college_mathematics", "college_physics", "econometrics",
    "high_school_statistics", "moral_scenarios",
    "astronomy", "college_computer_science", "elementary_mathematics", "formal_logic",
    "global_facts", "high_school_chemistry", "high_school_european_history",
    "high_school_geography", "high_school_macroeconomics", "high_school_physics",
    "high_school_us_history", "high_school_world_history", "international_law", "jurisprudence",
    "logical_fallacies", "machine_learning", "marketing", "philosophy", "prehistory", "sociology",
    "us_foreign_policy", "world_religions",
]
frames = []
for s in SUBJECTS:
    p = download(f"https://huggingface.co/datasets/cais/mmlu/resolve/main/{s}/test-00000-of-00001.parquet",
                 f"mmlu_{s}.parquet")
    df = pd.read_parquet(p)
    df["subject"] = s
    frames.append(df.sample(n=min(N_MMLU_PER_SUBJECT, len(df)), random_state=RANDOM_SEED))
mmlu = pd.concat(frames, ignore_index=True)
mmlu["qid"] = [f"mmlu_{i:04d}" for i in range(len(mmlu))]
for j, L in enumerate("ABCD"):
    mmlu[L] = mmlu["choices"].map(lambda c: str(c[j]))
mmlu["answer"] = mmlu["answer"].map(lambda a: "ABCD"[int(a)])
mmlu = mmlu[["qid", "subject", "question", "A", "B", "C", "D", "answer"]]
print(f"{len(mmlu):,} questions from {mmlu['subject'].nunique()} subjects")

SYSTEM = "You are an expert answering multiple-choice exam questions."


def mcq_prompt(row, order="ABCD", style="base"):
    """The question with options shown in `order` (a permutation of the original letters)."""
    shown = "\n".join(f"{'ABCD'[j]}. {row[orig]}" for j, orig in enumerate(order))
    if style == "base":
        return [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": f"Question: {row['question']}\n{shown}\n"
                                            "Answer with the letter of the correct option only."}]
    subject = row["subject"].replace("_", " ")
    return [{"role": "user", "content": f"Here is a question from a {subject} exam.\n\n"
                                        f"{row['question']}\n\n{shown}\n\nReply with A, B, C or D."}]


def letter_probs(resp):
    """Probabilities of the four answer letters from the first token's top log-probabilities.

    A letter that is not among the 20 returned tokens gets exactly 0 before renormalizing. That
    0 means "below the 20th token", not "impossible": the API reports nothing smaller. `floor`
    is the probability of the 20th token, an upper bound on any letter that is missing.
    """
    top = resp["choices"][0]["logprobs"]["content"][0]["top_logprobs"]
    p = dict.fromkeys("ABCD", 0.0)
    for t in top:
        tok = t["token"].strip().upper().rstrip(".)")
        if tok in p:
            p[tok] += math.exp(t["logprob"])
    mass = sum(p.values())
    floor = math.exp(min(t["logprob"] for t in top))
    return [p[L] / mass if mass > 0 else 0.25 for L in "ABCD"], mass, floor


def logprob_job(args):
    row, order, style, tag = args
    body = {"messages": mcq_prompt(row, order, style), "max_tokens": 1, "temperature": 0,
            "logprobs": True, "top_logprobs": 20, "seed": RANDOM_SEED, "user": tag}
    resp = call("chat", body)
    if filtered(resp):
        return [np.nan] * 4, np.nan, np.nan
    probs_shown, mass, floor = letter_probs(resp)
    probs = [0.0] * 4                       # map back to the original letters
    for j, orig in enumerate(order):
        probs["ABCD".index(orig)] = probs_shown[j]
    return probs, mass, floor


rng = np.random.default_rng(RANDOM_SEED)
orders = ["".join("ABCD"[k] for k in rng.permutation(4)) for _ in range(len(mmlu))]
rows = [r for _, r in mmlu.iterrows()]
variants = {
    "base": [(r, "ABCD", "base", "base") for r in rows],
    "rerun": [(r, "ABCD", "base", "rerun") for r in rows],      # same request, asked again
    "shuffled": [(r, o, "base", "shuffled") for r, o in zip(rows, orders)],
    "altprompt": [(r, "ABCD", "alt", "altprompt") for r in rows],
}
mcq = mmlu[["qid"]].copy()
for name, jobs in variants.items():
    out = run_all(jobs, logprob_job)
    for j, L in enumerate("ABCD"):
        mcq[f"{name}_p{L}"] = [o[0][j] for o in out]
    mcq[f"{name}_letter_mass"] = [o[1] for o in out]
    mcq[f"{name}_top20_floor"] = [o[2] for o in out]
    print(f"{name:>10}: done, median probability mass on the four letters "
          f"{np.nanmedian(mcq[f'{name}_letter_mass']):.4f}")
mcq["shuffled_order"] = orders

2,520 questions from 28 subjects


      base: done, median probability mass on the four letters 1.0000


     rerun: done, median probability mass on the four letters 1.0000


  shuffled: done, median probability mass on the four letters 1.0000


 altprompt: done, median probability mass on the four letters 0.9948


The `user` field differs between the base run and the re-run only so that the cache stores
them as two separate requests; it does not reach the prompt, but it does mean the two requests
are not byte-for-byte identical (notebook 16 calls them the same prompt and settings). Next, stated confidence and
sampled answers.

In [4]:
def verbal_job(row):
    msgs = mcq_prompt(row)
    msgs[-1] = {"role": "user", "content": msgs[-1]["content"].replace(
        "Answer with the letter of the correct option only.",
        'Reply in JSON as {"answer": "<letter>", "confidence": <0-100>}, where confidence is '
        "your probability, in percent, that your answer is correct.")}
    body = {"messages": msgs, "max_tokens": 30, "temperature": 0, "seed": RANDOM_SEED,
            "response_format": {"type": "json_object"}}
    resp = call("chat", body)
    if filtered(resp):
        return "", np.nan
    try:
        d = json.loads(resp["choices"][0]["message"]["content"])
        return str(d.get("answer", "")).strip().upper()[:1], float(d.get("confidence", np.nan))
    except (json.JSONDecodeError, ValueError, TypeError):
        return "", np.nan


def sample_job(row):
    body = {"messages": mcq_prompt(row), "max_tokens": 1, "temperature": 1.0,
            "n": N_MCQ_SAMPLES, "seed": RANDOM_SEED}
    resp = call("chat", body)
    if filtered(resp):
        return [np.nan] * 4, np.nan
    letters = [c["message"]["content"].strip().upper()[:1] for c in resp["choices"]]
    counts = Counter(letters)
    return [counts.get(L, 0) for L in "ABCD"], sum(v for k, v in counts.items() if k not in "ABCD")


verbal = run_all(rows, verbal_job)
mcq["verbal_answer"] = [v[0] for v in verbal]
mcq["verbal_confidence"] = [v[1] for v in verbal]
samples = run_all(rows, sample_job)
for j, L in enumerate("ABCD"):
    mcq[f"samples_{L}"] = [s[0][j] for s in samples]
mcq["samples_invalid"] = [s[1] for s in samples]

# Drop every question the content filter refused in ANY variant, so all variants describe the
# same questions. (moral_scenarios and some history questions can trip the filter.)
refused = mcq[[f"{v}_pA" for v in variants] + ["samples_A"]].isna().any(axis=1)
by_subject = mmlu.loc[refused.values, "subject"].value_counts().to_dict()
mcq, mmlu = mcq[~refused.values].reset_index(drop=True), mmlu[~refused.values].reset_index(drop=True)
print(f"{refused.sum()} questions dropped because the content filter refused at least one "
      f"request: {by_subject}")
print(f"{len(mmlu):,} questions kept; stated confidence parsed for "
      f"{mcq['verbal_confidence'].notna().mean():.1%}; {int(mcq['samples_invalid'].sum())} of "
      f"{N_MCQ_SAMPLES * len(mcq):,} samples were not a letter")

7 questions dropped because the content filter refused at least one request: {'high_school_us_history': 4, 'high_school_world_history': 3}
2,513 questions kept; stated confidence parsed for 100.0%; 85 of 25,130 samples were not a letter


## 2. Element profiles as tagged claims (PubChem periodic table)

In [5]:
table = pd.read_csv(download("https://pubchem.ncbi.nlm.nih.gov/rest/pug/periodictable/CSV",
                             "pubchem_periodic_table.csv"))
if LIMIT:
    table = table.head(LIMIT)
ATTRIBUTES = {
    "atomic_number": ("AtomicNumber", "integer"),
    "symbol": ("Symbol", "chemical symbol"),
    "atomic_mass": ("AtomicMass", "in u"),
    "standard_state": ("StandardState", "one of: Solid, Liquid, Gas"),
    "group_block": ("GroupBlock", "one of: Nonmetal, Noble gas, Alkali metal, Alkaline earth metal, "
                                  "Metalloid, Halogen, Post-transition metal, Transition metal, "
                                  "Lanthanide, Actinide"),
    "year_discovered": ("YearDiscovered", "a year, or Ancient"),
    "melting_point": ("MeltingPoint", "in kelvin"),
    "boiling_point": ("BoilingPoint", "in kelvin"),
    "density": ("Density", "in g/cm3"),
    "electronegativity": ("Electronegativity", "Pauling scale"),
    "ionization_energy": ("IonizationEnergy", "first ionization energy in eV"),
}
attr_lines = "\n".join(f"- {a}: {fmt}" for a, (_, fmt) in ATTRIBUTES.items())


def profile_body(name, temperature, seed):
    prompt = (f"Write a short factual profile of the chemical element {name} as a list of claims, "
              f"one claim per attribute below. For each claim give a one-sentence statement, the "
              f"value alone, and your confidence (0-100) that the value is correct.\n{attr_lines}\n"
              'Reply in JSON as {"claims": [{"attribute": ..., "sentence": ..., "value": ..., '
              '"confidence": ...}]}.')
    return {"messages": [{"role": "user", "content": prompt}], "max_tokens": 900,
            "temperature": temperature, "seed": seed, "response_format": {"type": "json_object"}}


def parse_claims(resp):
    if filtered(resp):
        return {}
    try:
        claims = json.loads(resp["choices"][0]["message"]["content"]).get("claims", [])
    except json.JSONDecodeError:
        return {}
    return {c.get("attribute"): c for c in claims if isinstance(c, dict) and c.get("attribute") in ATTRIBUTES}


def num(x):
    """The first number in a value, scientific notation included (8.988e-05 stays 8.988e-05)."""
    m = re.search(r"[-+]?(\d+\.?\d*|\.\d+)([eE][-+]?\d+)?", str(x).replace(",", ""))
    return float(m.group()) if m else None


def grade(attribute, claimed, truth):
    """True / False against PubChem, or None when PubChem has no value to check against."""
    if truth is None or (isinstance(truth, float) and math.isnan(truth)) or str(truth).strip() == "":
        return None
    c, t = str(claimed).strip().lower(), str(truth).strip().lower()
    if attribute in ("symbol", "standard_state", "group_block"):
        t = t.replace("expected to be a ", "")
        return c == t
    if attribute == "year_discovered":
        if t == "ancient":
            return "ancient" in c or (num(c) is not None and num(c) < 1600)
        return num(c) is not None and abs(num(c) - float(t)) <= 2
    if attribute == "atomic_number":                    # a whole number, no truncation
        return bool(re.fullmatch(r"\d+", c)) and int(c) == int(float(t))
    if not re.fullmatch(r"[-+]?(\d+\.?\d*|\.\d+)([eE][-+]?\d+)?", c.replace(",", "")):
        return None   # not a bare number in the requested unit: ungradable, never "true"
    # Relative tolerances. Atomic mass gets 1% because PubChem rounds some standard masses
    # (lithium is listed as 7.0, where textbooks print 6.94).
    tol = {"atomic_mass": 0.01, "melting_point": 0.05, "boiling_point": 0.05, "density": 0.05,
           "ionization_energy": 0.03}.get(attribute)
    if attribute == "electronegativity":
        return num(c) is not None and abs(num(c) - float(t)) <= 0.1
    return num(c) is not None and abs(num(c) - float(t)) <= tol * abs(float(t))


elements = list(table["Name"])
base = run_all([profile_body(n, 0, RANDOM_SEED) for n in elements], lambda b: parse_claims(call("chat", b)))
resamples = [run_all([profile_body(n, 1.0, RANDOM_SEED + i) for n in elements],
                     lambda b: parse_claims(call("chat", b))) for i in range(1, N_ELEMENT_RESAMPLES + 1)]

claim_rows = []
for i, (_, el) in enumerate(table.iterrows()):
    for attribute, (col, _) in ATTRIBUTES.items():
        c = base[i].get(attribute)
        if c is None or str(c.get("value", "")).strip().lower() in ("", "unknown", "n/a", "none"):
            continue
        truth = el[col]
        agree = sum(1 for r in resamples if r[i].get(attribute) is not None
                    and str(r[i][attribute].get("value")).strip().lower() == str(c.get("value")).strip().lower())
        claim_rows.append({"element": el["Name"], "atomic_number": int(el["AtomicNumber"]),
                           "attribute": attribute, "sentence": c.get("sentence", ""),
                           "claimed_value": str(c.get("value")), "true_value": "" if pd.isna(truth) else str(truth),
                           "correct": grade(attribute, c.get("value"), truth),
                           "stated_confidence": num(c.get("confidence")),
                           "resample_agreement": agree})
claims = pd.DataFrame(claim_rows)
gradable = claims["correct"].notna()
print(f"{len(claims):,} claims about {claims['element'].nunique()} elements; {gradable.sum():,} gradable "
      f"against PubChem; {claims.loc[gradable, 'correct'].astype(bool).mean():.1%} of those correct")

1,296 claims about 118 elements; 1,197 gradable against PubChem; 97.3% of those correct


## 3. Retrieval and short answers (HotpotQA distractor validation)

In [6]:
hp = pd.read_parquet(download(
    "https://huggingface.co/datasets/hotpotqa/hotpot_qa/resolve/main/distractor/validation-00000-of-00001.parquet",
    "hotpot_distractor_validation.parquet")).sample(n=N_HOTPOT, random_state=RANDOM_SEED)

questions, passages = [], []
for i, (_, r) in enumerate(hp.iterrows()):
    qid = f"hotpot_{i:04d}"
    gold = set(r["supporting_facts"]["title"])
    questions.append({"qid": qid, "question": r["question"], "answer": r["answer"],
                      "type": r["type"], "level": r["level"]})
    for j, (title, sents) in enumerate(zip(r["context"]["title"], r["context"]["sentences"])):
        passages.append({"qid": qid, "passage_idx": j, "title": title,
                         "text": "".join(sents).strip(), "is_supporting": title in gold})
questions, passages = pd.DataFrame(questions), pd.DataFrame(passages)


def embed_batch(texts):
    data = call("embed", {"input": texts})["data"]
    return [np.array(d["embedding"]) for d in sorted(data, key=lambda d: d["index"])]


def embed_all(texts, batch=64):
    out = run_all([texts[i:i + batch] for i in range(0, len(texts), batch)], embed_batch, workers=4)
    return np.vstack([v for b in out for v in b])


q_vec = embed_all(list(questions["question"]))
p_vec = embed_all([f"{t}: {x}" for t, x in zip(passages["title"], passages["text"])])
q_vec /= np.linalg.norm(q_vec, axis=1, keepdims=True)
p_vec /= np.linalg.norm(p_vec, axis=1, keepdims=True)
q_index = {q: k for k, q in enumerate(questions["qid"])}
passages["similarity"] = [float(p_vec[k] @ q_vec[q_index[q]]) for k, q in enumerate(passages["qid"])]
passages["rank"] = passages.groupby("qid")["similarity"].rank(ascending=False, method="first").astype(int)
print(f"{len(questions)} questions, {len(passages):,} passages embedded")


def answer_job(args):
    qid, k = args
    top = passages[(passages["qid"] == qid) & (passages["rank"] <= k)].sort_values("rank")
    context = "\n\n".join(f"[{t}] {x}" for t, x in zip(top["title"], top["text"]))
    q = questions.loc[q_index[qid], "question"]
    body = {"messages": [{"role": "user", "content":
            f"Answer the question using only the passages below. Reply with the short answer only "
            f"(a few words, or yes / no).\n\nPassages:\n{context}\n\nQuestion: {q}"}],
            "max_tokens": 20, "temperature": 0, "seed": RANDOM_SEED}
    resp = call("chat", body)
    return None if filtered(resp) else resp["choices"][0]["message"]["content"].strip()


def normalize(s):
    s = "".join(ch for ch in str(s).lower() if ch not in set(string.punctuation))
    return " ".join(w for w in s.split() if w not in ("a", "an", "the"))


def f1(pred, gold):
    """Token-overlap F1 as in HotpotQA's official evaluator (hotpot_evaluate_v1.py): a yes / no /
    noanswer answer scores 0 unless it matches the reference exactly after normalizing."""
    np_, ng = normalize(pred), normalize(gold)
    if (np_ in ("yes", "no", "noanswer") or ng in ("yes", "no", "noanswer")) and np_ != ng:
        return 0.0
    p, g = np_.split(), ng.split()
    common = sum((Counter(p) & Counter(g)).values())
    if common == 0:
        return 0.0
    prec, rec = common / len(p), common / len(g)
    return 2 * prec * rec / (prec + rec)


jobs = [(q, k) for q in questions["qid"] for k in K_GRID]
answers = pd.DataFrame(jobs, columns=["qid", "k"])
answers["model_answer"] = run_all(jobs, answer_job)
refused_q = set(answers.loc[answers["model_answer"].isna(), "qid"])   # drop the whole question
questions = questions[~questions["qid"].isin(refused_q)].reset_index(drop=True)
passages = passages[~passages["qid"].isin(refused_q)].reset_index(drop=True)
answers = answers[~answers["qid"].isin(refused_q)].reset_index(drop=True)
print(f"{len(refused_q)} HotpotQA questions dropped because the content filter refused an answer request")
gold = dict(zip(questions["qid"], questions["answer"]))
answers["exact_match"] = [normalize(a) == normalize(gold[q]) for q, a in zip(answers["qid"], answers["model_answer"])]
answers["f1"] = [f1(a, gold[q]) for q, a in zip(answers["qid"], answers["model_answer"])]
print(answers.groupby("k")[["exact_match", "f1"]].mean().round(3))

800 questions, 7,970 passages embedded


1 HotpotQA questions dropped because the content filter refused an answer request
    exact_match     f1
k                     
1         0.383  0.498
2         0.528  0.666
3         0.562  0.706
5         0.588  0.752
10        0.636  0.797


## 4. Write the files

In [7]:
mmlu.to_csv(OUT / "mmlu_questions.csv", index=False)
mcq.to_csv(OUT / "mmlu_llm_outputs.csv", index=False)   # full precision
table.to_csv(OUT / "pubchem_elements.csv", index=False)
claims.to_csv(OUT / "element_claims.csv", index=False)
questions.to_csv(OUT / "hotpot_questions.csv", index=False)
passages.to_csv(OUT / "hotpot_passages.csv", index=False)
answers.to_csv(OUT / "hotpot_answers.csv", index=False)
log = {"chat_deployment": CHAT, "embedding_deployment": EMBED, "api_version": API_VERSION,
       "random_seed": RANDOM_SEED, "generated_on": time.strftime("%Y-%m-%d"),
       "counts": {"mmlu_questions": len(mmlu), "element_claims": len(claims),
                  "hotpot_questions": len(questions), "hotpot_answers": len(answers)},
       "response_models": dict(RESPONSE_MODELS),
       "notes": ["Probabilities are stored at full precision; a letter probability of exactly 0 "
                 "means the letter was not among the 20 returned tokens (see *_top20_floor).",
                 "The base and rerun requests differ only in the API's user field.",
                 "The cache key is a hash of kind, deployment name and request body; it does not "
                 "include the endpoint or API version, which are recorded here instead.",
                 "HotpotQA F1 follows the official evaluator's yes/no/noanswer rule.",
                 "response_models counts responses as read during this run: a cached request "
                 "used twice counts twice, so the totals exceed the number of distinct requests."],
       "usage_this_run": dict(USAGE)}
(OUT / "generation_log.json").write_text(json.dumps(log, indent=2))
for f in sorted(OUT.glob("*.csv")):
    print(f"{f.name:<26} {f.stat().st_size / 1e6:6.2f} MB")
print(json.dumps(log["usage_this_run"], indent=2))

element_claims.csv           0.13 MB
hotpot_answers.csv           0.17 MB
hotpot_passages.csv          4.94 MB
hotpot_questions.csv         0.11 MB
mmlu_llm_outputs.csv         1.17 MB
mmlu_questions.csv           1.12 MB
pubchem_elements.csv         0.01 MB
{
  "cached": 19966
}
